# Repelón Risk Lens — V4 · 2/2: Análisis climático y comparaciones

**Ejecutar después del notebook 1**, que genera `predios_para_demo_chirps_2026-11-15.csv`. Este notebook inicia su propia sesión y carga sus propias variables.

Contiene: NASA POWER 2018–2025, ET0, escenario hipotético de yuca, calendario CHIRPS+NDVI y comparaciones CHIRPS/GPM/ERA5. **Ninguna comparación altera el índice principal del notebook 1.**


## 0. Preparación y lectura de resultados territoriales

In [ ]:
# Configuración independiente: no usa variables del notebook 1
import os, json, shutil, requests
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import ee, geemap
from google.colab import drive

PROJECT_ID = 'agro-513620'
RUTA_BASE = '/content/drive/MyDrive/Agro'
FECHA_FIN = '2026-11-15'
LLUVIA_INI_STR, LLUVIA_FIN_STR = '2026-09-16', '2026-10-31'
ANIOS_CLIMA, TAMANO_LOTE = 10, 500
ESCALA_GPM, ESCALA_CHIRPS = 11132, 5566
CHIRPS_ID = 'UCSB-CHC/CHIRPS/V3/PENTAD'
IMERG = 'NASA/GPM_L3/IMERG_V07'
PESO_AMENAZA = 0.5
NUBOSIDAD_MAX = 30

drive.mount('/content/drive')
try:
    ee.Initialize(project=PROJECT_ID)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

# Se necesita la geometría para las consultas a Earth Engine y para la latitud de ET0.
ruta_drive = f'{RUTA_BASE}/repelon_predios_rurales.gpkg'
ruta_local = '/content/repelon_predios_rurales.gpkg'
shutil.copy2(ruta_drive, ruta_local)
gdf = gpd.read_file(ruta_local)
gdf['CODIGO'] = gdf['CODIGO'].astype(str).str.strip()
gdf = (gdf.assign(_area=gdf.geometry.area)
       .sort_values('_area', ascending=False)
       .drop_duplicates('CODIGO').drop(columns='_area').reset_index(drop=True))
assert len(gdf) == 1848 and gdf['CODIGO'].is_unique

# Leer resultados reales del notebook 1, sin recalcular el riesgo.
ruta_demo = f'{RUTA_BASE}/predios_para_demo_chirps_{FECHA_FIN}.csv'
assert os.path.exists(ruta_demo), ('Ejecuta primero el notebook 1: falta ' + ruta_demo)
demo = pd.read_csv(ruta_demo, dtype={'CODIGO':str})
assert len(demo) == 1848 and demo['CODIGO'].is_unique
cols = ['CODIGO','V_Vulnerability','precip_mm','precip_normal_mm','Amenaza_Norm','Riesgo_Index','Nivel_Riesgo']
gdf = gdf.merge(demo[cols], on='CODIGO', how='left', validate='one_to_one')
assert gdf['Riesgo_Index'].notna().all(), 'Faltan riesgos del notebook 1'

def f(fecha):
    return pd.Timestamp(fecha).strftime('%Y-%m-%d')

LLUVIA_INI, LLUVIA_FIN = pd.Timestamp(LLUVIA_INI_STR), pd.Timestamp(LLUVIA_FIN_STR)
UN_DIA = pd.Timedelta(days=1)
dias_periodo = (LLUVIA_FIN - LLUVIA_INI).days + 1  # 46 días: octubre 26-31 = 6
FIN = pd.Timestamp(FECHA_FIN)
x0,y0,x1,y1 = gdf.to_crs(4326).total_bounds
region = ee.Geometry.Rectangle([float(x0),float(y0),float(x1),float(y1)])
cent = gdf[['CODIGO','geometry']].copy()
cent['geometry'] = cent.geometry.centroid
cent = cent.to_crs(4326)

print(f'Preparado: {len(gdf)} predios, periodo CHIRPS {LLUVIA_INI_STR}–{LLUVIA_FIN_STR} ({dias_periodo} días)')


## 1. NASA POWER: obtener datos diarios 2018–2025

In [ ]:
# Descarga NASA POWER 2018-2025 (faltaba por completo en v3)
# Se guarda una copia CSV en Drive; en ejecuciones posteriores se reutiliza.
RUTA_POWER_CACHE = f'{RUTA_BASE}/nasa_power_diario_repelon_2018_2025.csv'
PARAM_POWER = ['T2M_MAX','T2M_MIN','T2MDEW','WS2M',
               'ALLSKY_SFC_SW_DWN','PRECTOTCORR','PS']

if os.path.exists(RUTA_POWER_CACHE):
    datos_power = pd.read_csv(RUTA_POWER_CACHE, dtype={'FECHA':str})
    print('NASA POWER: datos cargados de CSV local de Drive')
else:
    centro = gdf.to_crs(4326).geometry.union_all().centroid
    url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
    params = {'parameters': ','.join(PARAM_POWER), 'community':'AG',
              'longitude':round(float(centro.x),5), 'latitude':round(float(centro.y),5),
              'start':'20180101', 'end':'20251231', 'format':'JSON'}
    respuesta = requests.get(url, params=params, timeout=120)
    respuesta.raise_for_status()
    payload = respuesta.json()
    assert 'properties' in payload and 'parameter' in payload['properties'], 'Respuesta inesperada de NASA POWER'
    datos_power = pd.DataFrame(payload['properties']['parameter'])
    datos_power.index.name = 'FECHA'
    datos_power = datos_power.reset_index()
    datos_power.to_csv(RUTA_POWER_CACHE, index=False)
    print('NASA POWER: descargado y guardado en Drive')

assert set(PARAM_POWER).issubset(datos_power.columns), 'Faltan variables NASA POWER'
# Mantener la estructura esperada por la celda 13 original (parámetros como filas)
df_power = datos_power.set_index('FECHA').T
print('Variables:', list(df_power.index))
print('Días:', len(df_power.columns))


## 13. NASA POWER — Datos meteorológicos diarios para Repelón

In [ ]:

# Convertir NASA POWER a formato diario
df_power_diario = df_power.T.copy()

df_power_diario.index.name = "FECHA"
df_power_diario = df_power_diario.reset_index()

df_power_diario["FECHA"] = pd.to_datetime(
    df_power_diario["FECHA"].astype(str),
    format="%Y%m%d"
)

# Convertir variables a valores numéricos
columnas = [
    "T2M_MAX", "T2M_MIN", "T2MDEW", "WS2M",
    "ALLSKY_SFC_SW_DWN", "PRECTOTCORR", "PS"
]

df_power_diario[columnas] = df_power_diario[columnas].apply(
    pd.to_numeric, errors="coerce"
).replace(-999, np.nan)
assert len(df_power_diario) == 2922, "Se esperaban 2922 días entre 2018 y 2025"
assert not df_power_diario[columnas].isna().any().any(), "NASA POWER tiene valores faltantes; revisar antes de ET0"

print("Dimensiones:", df_power_diario.shape)
print("Primera fecha:", df_power_diario["FECHA"].min())
print("Última fecha:", df_power_diario["FECHA"].max())
print("Fechas duplicadas:", df_power_diario["FECHA"].duplicated().sum())

display(df_power_diario.head())


## 13.1 Validación y guardado de los datos NASA POWER.

In [ ]:
print("Valores faltantes:")
print(df_power_diario.isna().sum())

print("\nResumen de variables:")
display(df_power_diario.describe().round(2))

# Guardar la tabla corregida
ruta_power = f"{RUTA_BASE}/power_diario_repelon_2018_2025.csv"
df_power_diario.to_csv(ruta_power, index=False)

print("\nArchivo guardado:", ruta_power)

## 13.2. Cálculo de evapotranspiración de referencia (ET0) con FAO-56 Penman-Monteith.

In [ ]:

# ============================================================
# 13.2. CALCULO DE ET0 - FAO-56 PENMAN-MONTEITH
# ============================================================

import numpy as np
import pandas as pd

# 1. Obtener latitud del centro del conjunto de predios
centro = gdf.geometry.union_all().centroid

if gdf.crs is None:
    raise ValueError("El GeoDataFrame no tiene CRS definido.")

centro_geo = gpd.GeoSeries(
    [centro], crs=gdf.crs
).to_crs("EPSG:4326")

latitud = float(centro_geo.iloc[0].y)

print(f"Latitud utilizada: {latitud:.5f}°")

# 2. Preparar variables meteorologicas
df_et0 = df_power_diario.copy()

tmax = df_et0["T2M_MAX"]
tmin = df_et0["T2M_MIN"]
tmean = (tmax + tmin) / 2

tdew = df_et0["T2MDEW"]
viento = df_et0["WS2M"]
rs = df_et0["ALLSKY_SFC_SW_DWN"]
presion = df_et0["PS"]

# 3. Presion de vapor de saturacion y real (kPa)
def presion_vapor(T):
    return 0.6108 * np.exp((17.27 * T) / (T + 237.3))

es = (presion_vapor(tmax) + presion_vapor(tmin)) / 2
ea = presion_vapor(tdew)

# 4. Pendiente de la curva de presion de vapor
delta = (
    4098 * presion_vapor(tmean)
    / ((tmean + 237.3) ** 2)
)

# 5. Radiacion extraterrestre (FAO-56)
J = df_et0["FECHA"].dt.dayofyear

phi = np.radians(latitud)

dr = 1 + 0.033 * np.cos(2 * np.pi * J / 365)

declinacion = (
    0.409 * np.sin(2 * np.pi * J / 365 - 1.39)
)

ws = np.arccos(
    np.clip(-np.tan(phi) * np.tan(declinacion), -1, 1)
)

Gsc = 0.0820

ra = (
    (24 * 60 / np.pi)
    * Gsc
    * dr
    * (
        ws * np.sin(phi) * np.sin(declinacion)
        + np.cos(phi) * np.cos(declinacion) * np.sin(ws)
    )
)

# 6. Radiacion neta (MJ/m2/dia)
# Estimacion de elevacion a partir de la presion atmosferica
p_media = presion.mean()

elevacion = 44307.7 * (
    1 - (p_media / 101.3) ** 0.1903
)

rso = (0.75 + 2e-5 * elevacion) * ra

rns = 0.77 * rs

relacion_radiacion = np.clip(
    rs / rso.replace(0, np.nan), 0.3, 1.0
)

sigma = 4.903e-9

rnl = (
    sigma
    * ((tmax + 273.16)**4 + (tmin + 273.16)**4) / 2
    * (0.34 - 0.14 * np.sqrt(ea))
    * (1.35 * relacion_radiacion - 0.35)
)

rn = rns - rnl

# 7. Constante psicrometrica
gamma = 0.000665 * presion

# 8. ET0 diaria - FAO-56 Penman-Monteith
numerador = (
    0.408 * delta * rn
    + gamma * (900 / (tmean + 273)) * viento * (es - ea)
)

denominador = delta + gamma * (1 + 0.34 * viento)

df_et0["ET0_mm_dia"] = numerador / denominador

# Evitar valores negativos no fisicos
df_et0["ET0_mm_dia"] = df_et0["ET0_mm_dia"].clip(lower=0)

# 9. Revisar resultados
print("\nElevacion estimada:", round(elevacion, 2), "m")
print("\nResumen de ET0:")
display(df_et0["ET0_mm_dia"].describe().round(2))

print("\nPrimeros registros:")
display(df_et0[["FECHA", "ET0_mm_dia"]].head())

# 10. Guardar resultados
ruta_et0 = f"{RUTA_BASE}/et0_fao56_repelon_2018_2025.csv"

df_et0.to_csv(ruta_et0, index=False)

print("\nArchivo guardado:", ruta_et0)


## 14. Estimación de la demanda hídrica de los cultivos (ETc)

In [ ]:

# ============================================================
# 14.1. PARAMETROS DE CULTIVO - YUCA
# Fuente: FAO-56, Tablas 11 y 12
# ============================================================

import pandas as pd

parametros_yuca = pd.DataFrame({
    "etapa": [
        "Inicial",
        "Desarrollo",
        "Media",
        "Final"
    ],
    "duracion_dias": [20, 40, 90, 60],
    "kc_inicio": [0.30, 0.30, 0.80, 0.80],
    "kc_final": [0.30, 0.80, 0.80, 0.30]
})

print("PARAMETROS DE REFERENCIA - YUCA")
display(parametros_yuca)

print("Duracion total del ciclo:",
      parametros_yuca["duracion_dias"].sum(), "dias")

print("Kc inicial:", 0.30)
print("Kc medio:", 0.80)
print("Kc final:", 0.30)

# Guardar parametros
ruta_kc = f"{RUTA_BASE}/parametros_kc_yuca.csv"
parametros_yuca.to_csv(ruta_kc, index=False)

print("\nArchivo guardado:", ruta_kc)


## 14.2 construir la curva diaria de Kc y calcular ETc.

In [ ]:

# ============================================================
# 14.2. CURVA DIARIA DE Kc Y CALCULO DE ETc - YUCA
# ============================================================

import numpy as np
import pandas as pd

# 1. Copiar la serie de ET0
df_etc = df_et0[["FECHA", "ET0_mm_dia"]].copy()
df_etc["FECHA"] = pd.to_datetime(df_etc["FECHA"])

# 2. Definir el escenario de siembra
# Fecha hipotetica: 1 de mayo de cada año
fecha_siembra = {
    anio: pd.Timestamp(year=anio, month=5, day=1)
    for anio in df_etc["FECHA"].dt.year.unique()
}

# 3. Construir la curva diaria de Kc
def calcular_kc(dia):
    if dia <= 20:
        return 0.30, "Inicial"

    elif dia <= 60:
        kc = np.linspace(0.30, 0.80, 40)[dia - 21]
        return kc, "Desarrollo"

    elif dia <= 150:
        return 0.80, "Media"

    elif dia <= 210:
        kc = np.linspace(0.80, 0.30, 60)[dia - 151]
        return kc, "Final"

    return np.nan, None


# 4. Asignar etapa y Kc a cada fecha
df_etc["FECHA_SIEMBRA"] = (
    df_etc["FECHA"].dt.year.map(fecha_siembra)
)

df_etc["DIA_CULTIVO"] = (
    df_etc["FECHA"] - df_etc["FECHA_SIEMBRA"]
).dt.days + 1

resultados = df_etc["DIA_CULTIVO"].apply(
    lambda dia: calcular_kc(dia) if 1 <= dia <= 210
    else (np.nan, None)
)

df_etc["Kc"] = resultados.apply(lambda x: x[0])
df_etc["ETAPA"] = resultados.apply(lambda x: x[1])

# 5. Calcular evapotranspiracion del cultivo
df_etc["ETc_mm_dia"] = (
    df_etc["ET0_mm_dia"] * df_etc["Kc"]
)

# 6. Revisar resultados
print("Registros totales:", len(df_etc))
print("Dias dentro del ciclo:", df_etc["Kc"].notna().sum())

print("\nDistribucion por etapas:")
display(df_etc["ETAPA"].value_counts())

print("\nResumen de Kc y ETc:")
display(
    df_etc[["Kc", "ETc_mm_dia"]]
    .describe()
    .round(3)
)

print("\nPrimeros registros del ciclo:")
display(
    df_etc[df_etc["Kc"].notna()][
        ["FECHA", "DIA_CULTIVO", "ETAPA", "Kc",
         "ET0_mm_dia", "ETc_mm_dia"]
    ].head(25)
)

# 7. Guardar resultados
ruta_etc = f"{RUTA_BASE}/etc_yuca_2018_2025.csv"
df_etc.to_csv(ruta_etc, index=False)

print("\nArchivo guardado:", ruta_etc)


## 2. Balance climático hipotético yuca (no balance de suelo)

In [ ]:

# ============================================================
# 14.3. BALANCE CLIMATICO DIARIO - YUCA
# Precipitacion NASA POWER vs. demanda ETc
# ============================================================

# 1. Preparar las series diarias
df_balance = pd.merge(
    df_etc[[
        "FECHA", "FECHA_SIEMBRA", "DIA_CULTIVO",
        "ETAPA", "Kc", "ET0_mm_dia", "ETc_mm_dia"
    ]],
    df_power_diario[["FECHA", "PRECTOTCORR"]],
    on="FECHA",
    how="left",
    validate="one_to_one"
)

# 2. Renombrar precipitacion
df_balance = df_balance.rename(
    columns={"PRECTOTCORR": "P_mm_dia"}
)

# 3. Calcular balance solamente durante el ciclo
# Balance positivo: lluvia superior a la demanda
# Balance negativo: demanda superior a la lluvia

df_balance["balance_climatico_mm"] = (
    df_balance["P_mm_dia"] - df_balance["ETc_mm_dia"]
)

df_balance["deficit_mm"] = (
    df_balance["ETc_mm_dia"] - df_balance["P_mm_dia"]
).clip(lower=0)

df_balance["excedente_mm"] = (
    df_balance["P_mm_dia"] - df_balance["ETc_mm_dia"]
).clip(lower=0)

# 4. Calcular deficit acumulado durante cada ciclo
df_balance["deficit_acumulado_mm"] = (
    df_balance.groupby("FECHA_SIEMBRA")["deficit_mm"]
    .cumsum()
)

# 5. Filtrar dias activos del cultivo
df_ciclo = df_balance[df_balance["ETc_mm_dia"].notna()].copy()

# 6. Revisar resultados
print("DIAS ANALIZADOS:", len(df_ciclo))

print("\nResumen del balance climatico:")
display(
    df_ciclo[[
        "P_mm_dia",
        "ETc_mm_dia",
        "balance_climatico_mm",
        "deficit_mm",
        "excedente_mm"
    ]].describe().round(2)
)

print("\nDias con demanda superior a la lluvia:",
      (df_ciclo["deficit_mm"] > 0).sum())

print("Dias con lluvia superior o igual a la demanda:",
      (df_ciclo["balance_climatico_mm"] >= 0).sum())

# 7. Resumen por ciclo de cultivo
resumen_ciclos = (
    df_ciclo.groupby("FECHA_SIEMBRA")
    .agg(
        lluvia_total_mm=("P_mm_dia", "sum"),
        demanda_total_mm=("ETc_mm_dia", "sum"),
        deficit_acumulado_mm=("deficit_mm", "sum"),
        excedente_acumulado_mm=("excedente_mm", "sum"),
        dias_con_deficit=("deficit_mm", lambda x: (x > 0).sum())
    )
    .round(2)
)

print("\nRESULTADOS POR CICLO:")
display(resumen_ciclos)

# 8. Guardar resultados
ruta_balance = f"{RUTA_BASE}/balance_climatico_yuca_2018_2025.csv"

df_balance.to_csv(ruta_balance, index=False)

ruta_resumen = f"{RUTA_BASE}/resumen_balance_yuca_2018_2025.csv"

resumen_ciclos.to_csv(ruta_resumen)

print("\nArchivos guardados:")
print(ruta_balance)
print(ruta_resumen)


## 14.4

In [ ]:

# ============================================================
# 14.4. IDENTIFICACION DE EPISODIOS DE DEFICIT HIDRICO
# ============================================================

# 1. Preparar datos
df_eventos = df_balance[
    df_balance["ETc_mm_dia"].notna()
].copy()

df_eventos = df_eventos.sort_values(
    ["FECHA_SIEMBRA", "FECHA"]
).reset_index(drop=True)

# 2. Identificar episodios consecutivos de deficit
eventos = []

for fecha_ciclo, grupo in df_eventos.groupby("FECHA_SIEMBRA"):

    grupo = grupo.sort_values("FECHA").copy()

    grupo["dia_deficit"] = (
        grupo["deficit_mm"] > 0
    )

    grupo["grupo_evento"] = (
        grupo["dia_deficit"].ne(
            grupo["dia_deficit"].shift()
        ).cumsum()
    )

    episodios = grupo[grupo["dia_deficit"]].groupby(
        "grupo_evento"
    )

    for _, episodio in episodios:

        eventos.append({
            "FECHA_SIEMBRA": fecha_ciclo,
            "inicio": episodio["FECHA"].min(),
            "fin": episodio["FECHA"].max(),
            "duracion_dias": len(episodio),
            "deficit_acumulado_mm": episodio["deficit_mm"].sum(),
            "deficit_maximo_diario_mm": episodio["deficit_mm"].max()
        })

df_episodios = pd.DataFrame(eventos)

# 3. Resumen de episodios por ciclo
resumen_episodios = (
    df_episodios.groupby("FECHA_SIEMBRA")
    .agg(
        numero_episodios=("duracion_dias", "count"),
        duracion_media_dias=("duracion_dias", "mean"),
        episodio_mas_largo_dias=("duracion_dias", "max"),
        deficit_mayor_episodio_mm=("deficit_acumulado_mm", "max"),
        deficit_total_episodios_mm=("deficit_acumulado_mm", "sum")
    )
    .round(2)
)

print("RESUMEN DE EPISODIOS POR CICLO:")
display(resumen_episodios)

# 4. Mostrar los episodios mas prolongados
print("\n10 EPISODIOS MAS PROLONGADOS:")
display(
    df_episodios.sort_values(
        "duracion_dias", ascending=False
    ).head(10).round(2)
)

# 5. Distribucion mensual del deficit
df_eventos["MES"] = (
    df_eventos["FECHA"].dt.to_period("M").astype(str)
)

resumen_mensual = (
    df_eventos.groupby(["FECHA_SIEMBRA", "MES"])
    .agg(
        lluvia_mm=("P_mm_dia", "sum"),
        demanda_etc_mm=("ETc_mm_dia", "sum"),
        deficit_mm=("deficit_mm", "sum"),
        dias_con_deficit=("deficit_mm", lambda x: (x > 0).sum())
    )
    .round(2)
    .reset_index()
)

print("\nDISTRIBUCION MENSUAL:")
display(resumen_mensual)

# 6. Guardar resultados
ruta_episodios = (
    f"{RUTA_BASE}/episodios_deficit_yuca_2018_2025.csv"
)

ruta_mensual = (
    f"{RUTA_BASE}/deficit_mensual_yuca_2018_2025.csv"
)

df_episodios.to_csv(ruta_episodios, index=False)
resumen_mensual.to_csv(ruta_mensual, index=False)

print("\nArchivos guardados:")
print(ruta_episodios)
print(ruta_mensual)


## 14.5. Indicador de exposición al déficit hídrico

In [ ]:

# ============================================================
# 14.5. INDICADOR EXPLORATORIO DE EXPOSICION HIDRICA
# ============================================================

# 1. Resumir los indicadores por ciclo
indicador = resumen_episodios.copy()

indicador["deficit_total_mm"] = (
    resumen_ciclos["deficit_acumulado_mm"]
)

indicador["dias_con_deficit"] = (
    resumen_ciclos["dias_con_deficit"]
)

# 2. Calcular frecuencia de dias con deficit
indicador["frecuencia_pct"] = (
    indicador["dias_con_deficit"] / 210 * 100
)

# 3. Normalizar los tres componentes entre 0 y 100
columnas = [
    "frecuencia_pct",
    "deficit_total_mm",
    "episodio_mas_largo_dias"
]

for columna in columnas:
    minimo = indicador[columna].min()
    maximo = indicador[columna].max()

    if maximo > minimo:
        indicador[columna + "_norm"] = (
            (indicador[columna] - minimo)
            / (maximo - minimo) * 100
        )
    else:
        indicador[columna + "_norm"] = 0

# 4. Calcular indicador compuesto
indicador["exposicion_hidrica"] = (
    indicador["frecuencia_pct_norm"]
    + indicador["deficit_total_mm_norm"]
    + indicador["episodio_mas_largo_dias_norm"]
) / 3

# 5. Ordenar resultados
indicador = indicador.sort_values(
    "exposicion_hidrica", ascending=False
)

print("EXPOSICION HIDRICA POR CICLO")
display(
    indicador[[
        "frecuencia_pct",
        "deficit_total_mm",
        "episodio_mas_largo_dias",
        "exposicion_hidrica"
    ]].round(2)
)

# 6. Guardar resultados
ruta_indicador = (
    f"{RUTA_BASE}/indicador_exposicion_hidrica_yuca.csv"
)

indicador.to_csv(ruta_indicador)

print("\nArchivo guardado:", ruta_indicador)


## 2.5 Visualización de exposición

In [ ]:

import matplotlib.pyplot as plt

# Ordenar ciclos de mayor a menor exposición
grafica = indicador.sort_values(
    "exposicion_hidrica", ascending=True
).copy()

# Usar el año como etiqueta
etiquetas = grafica.index.strftime("%Y") if hasattr(
    grafica.index, "strftime"
) else grafica.index.astype(str)

plt.figure(figsize=(10, 5))

barras = plt.barh(
    etiquetas,
    grafica["exposicion_hidrica"]
)

plt.xlabel("Exposición hídrica (0-100)")
plt.ylabel("Ciclo de cultivo")
plt.title("Exposición hídrica estimada de la yuca por ciclo")
plt.xlim(0, 110)
plt.grid(axis="x", alpha=0.25)

for barra, valor in zip(barras, grafica["exposicion_hidrica"]):
    plt.text(
        valor + 1,
        barra.get_y() + barra.get_height() / 2,
        f"{valor:.2f}",
        va="center"
    )

plt.tight_layout()
plt.show()

print(
    "El ciclo con mayor exposición fue:",
    indicador["exposicion_hidrica"].idxmax()
)


Los resultados muestran una mayor exposición hídrica estimada durante los ciclos simulados de 2021 (100,00) y 2020 (69,46). Esta diferencia refleja la combinación de una mayor frecuencia de días con déficit, mayor déficit acumulado y episodios más prolongados. Los resultados permiten identificar períodos climáticos potencialmente desfavorables para un cultivo de referencia de yuca, bajo una fecha hipotética de siembra del 1 de mayo. No representan mediciones de estrés hídrico ni pérdidas agrícolas reales.

## 3. Calendario mensual CHIRPS y NDVI (descriptivo)

In [ ]:
# Calendario CHIRPS + NDVI; NO entra al riesgo
def enmascarar_s2(img):
    scl = img.select('SCL')
    return img.updateMask(scl.eq(4).Or(scl.eq(5)))

# ---- Parámetros del calendario ----
ANIO_INI_CAL = 2021
ANIO_FIN_CAL = 2025
ESCALA_CAL = 5566          # metros; escala aproximada de CHIRPS


def con_ndvi(img):
    """NDVI de una escena Sentinel-2 ya enmascarada (sin nubes ni sombras).
    No hace falta dividir entre 10000: el NDVI es una razón y la escala se cancela."""
    m = enmascarar_s2(img)
    nir, red = m.select('B8'), m.select('B4')
    return nir.subtract(red).divide(nir.add(red)).rename('NDVI')


def fila_mes(anio, mes):
    """Lluvia (mm) y NDVI medios de la zona en un mes."""
    ini = ee.Date.fromYMD(anio, mes, 1)
    fin = ini.advance(1, 'month')
    ndvi_m = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(region).filterDate(ini, fin)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
        .map(con_ndvi).median()
    )
    # CHIRPS pentadal: se asignan pentadas por su fecha de inicio; meses completos.
    lluvia_m = (
        ee.ImageCollection(CHIRPS_ID).filterDate(ini, fin)
        .select('precipitation').sum().rename('lluvia_mm')
    )
    vals = lluvia_m.addBands(ndvi_m).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=ESCALA_CAL,
        maxPixels=1e9, tileScale=4
    )
    return ee.Feature(None, vals).set('anio', anio).set('mes', mes)


filas = []
for anio in range(ANIO_INI_CAL, ANIO_FIN_CAL + 1):
    fc_anio = ee.FeatureCollection(
        ee.List.sequence(1, 12).map(lambda m: fila_mes(anio, m))
    )
    filas += [x['properties'] for x in fc_anio.getInfo()['features']]
    print(f'  año {anio} listo')

df_cal = pd.DataFrame(filas)
for col in ['NDVI', 'lluvia_mm']:
    if col not in df_cal.columns:
        df_cal[col] = float('nan')
df_cal = df_cal.sort_values(['anio', 'mes']).reset_index(drop=True)
df_cal.to_csv(f'{RUTA_BASE}/calendario_mensual_chirps_serie.csv', index=False)
print(f"Meses sin NDVI (sin escenas útiles): {df_cal['NDVI'].isna().sum()} de {len(df_cal)}")

clima = df_cal.groupby('mes')[['NDVI', 'lluvia_mm']].mean().round(3)
clima.to_csv(f'{RUTA_BASE}/calendario_mensual_chirps.csv')
print(clima)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(clima.index, clima['lluvia_mm'], color='dodgerblue', alpha=0.5)
ax.set_xlabel('Mes del año')
ax.set_ylabel('Lluvia mensual promedio (mm)')
ax.set_xticks(range(1, 13))
ax2 = ax.twinx()
ax2.plot(clima.index, clima['NDVI'], color='forestgreen', marker='o')
ax2.set_ylabel('NDVI')
plt.title(f'Calendario de la zona: lluvia (barras) y verdor (línea), {ANIO_INI_CAL}-{ANIO_FIN_CAL}')
plt.tight_layout()
plt.savefig(f'{RUTA_BASE}/calendario_mensual_chirps.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Comparación mensual GPM vs ERA5-Land

In [ ]:
# Comparación mensual de lluvia: GPM IMERG vs ERA5-Land (promedio de la zona, mm por mes)
ERA5 = 'ECMWF/ERA5_LAND/MONTHLY_AGGR'
N_MESES = 60   # enero 2021 a diciembre 2025


def fila_mes(i):
    ini = ee.Date.fromYMD(2021, 1, 1).advance(ee.Number(i), 'month')
    fin = ini.advance(1, 'month')
    gpm = (ee.ImageCollection(IMERG).filterDate(ini, fin).select('precipitation')
           .sum().multiply(0.5).rename('gpm_mm'))
    era = (ee.ImageCollection(ERA5).filterDate(ini, fin).first()
           .select('total_precipitation_sum').multiply(1000).rename('era5_mm'))   # metros -> mm
    valores = gpm.addBands(era).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=region, scale=ESCALA_GPM, maxPixels=1e9)
    return ee.Feature(None, valores.set('mes', ini.format('YYYY-MM')))


print('Consultando 60 meses (puede tardar 1-2 minutos)...')
info = ee.FeatureCollection(ee.List.sequence(0, N_MESES - 1).map(fila_mes)).getInfo()
cmp_ll = pd.DataFrame([x['properties'] for x in info['features']]).sort_values('mes').reset_index(drop=True)
cmp_ll.to_csv(f'{RUTA_BASE}/comparacion_lluvia_gpm_era5.csv', index=False)

corr = cmp_ll['gpm_mm'].corr(cmp_ll['era5_mm'])
anual = cmp_ll.assign(anio=cmp_ll['mes'].str[:4]).groupby('anio')[['gpm_mm', 'era5_mm']].sum().round(0)
print(f'\nCorrelación mensual GPM vs ERA5-Land: {corr:.2f}  (1 = idénticas)')
print(f"Promedio mensual: GPM {cmp_ll['gpm_mm'].mean():.0f} mm | ERA5-Land {cmp_ll['era5_mm'].mean():.0f} mm")
print('\nLluvia anual (mm):')
print(anual)

# Los dos meses de dic-2024 y ene-2025: ¿también ERA5 ve el evento?
print('\nDiciembre 2024 y enero 2025:')
print(cmp_ll[cmp_ll['mes'].isin(['2024-12', '2025-01'])].round(1))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(cmp_ll['mes'], cmp_ll['gpm_mm'], label='GPM IMERG (NASA)', color='dodgerblue')
ax.plot(cmp_ll['mes'], cmp_ll['era5_mm'], label='ERA5-Land (ECMWF)', color='darkorange')
ax.set_xticks(range(0, N_MESES, 6))
ax.set_xticklabels(cmp_ll['mes'][::6], rotation=45)
ax.set_ylabel('mm por mes')
ax.set_title(f'Lluvia mensual de la zona: dos fuentes independientes (correlación {corr:.2f})')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RUTA_BASE}/comparacion_lluvia_gpm_era5.png', dpi=200, bbox_inches='tight')
plt.show()

## 4b. Comparación mensual CHIRPS vs GPM vs ERA5-Land

In [ ]:
def fila_mes_chirps(i):
    ini = ee.Date.fromYMD(2021, 1, 1).advance(ee.Number(i), 'month')
    fin = ini.advance(1, 'month')
    ch = (ee.ImageCollection(CHIRPS_ID).filterDate(ini, fin).select('precipitation')
          .sum().rename('chirps_mm'))
    valores = ch.reduceRegion(reducer=ee.Reducer.mean(), geometry=region, scale=ESCALA_CHIRPS, maxPixels=1e9)
    return ee.Feature(None, valores.set('mes', ini.format('YYYY-MM')))


info = ee.FeatureCollection(ee.List.sequence(0, N_MESES - 1).map(fila_mes_chirps)).getInfo()
cmp_ch = pd.DataFrame([x['properties'] for x in info['features']]).sort_values('mes').reset_index(drop=True)
cmp3 = cmp_ll.merge(cmp_ch, on='mes')
cmp3.to_csv(f'{RUTA_BASE}/comparacion_lluvia_gpm_chirps_era5.csv', index=False)
print('Correlaciones mensuales (2021-2025):')
print(cmp3[['gpm_mm', 'chirps_mm', 'era5_mm']].corr().round(2))
print('\nPromedio mensual (mm):'); print(cmp3[['gpm_mm', 'chirps_mm', 'era5_mm']].mean().round(0))
print('\nDiciembre 2024 y enero 2025:'); print(cmp3[cmp3['mes'].isin(['2024-12', '2025-01'])].round(1))
print('\nLectura: si CHIRPS y GPM difieren de forma sistemática, la amenaza depende de la fuente (ver sección 15).')

## 5. GPM vs CHIRPS: mismas fechas, mismo riesgo base; GPM solo comparación

In [ ]:

def lluvia_gpm_mm(ini, fin_incl):
    """Lluvia acumulada GPM (mm). IMERG viene en mm/hora cada 30 min -> se multiplica por 0.5 h."""
    col = (ee.ImageCollection(IMERG).filterDate(f(ini), f(fin_incl + UN_DIA)).select('precipitation'))
    return col.sum().multiply(0.5).rename('gpm_mm'), col.size()


gpm_actual, n_gpm = lluvia_gpm_mm(LLUVIA_INI, LLUVIA_FIN)
esperadas_gpm = dias_periodo * 48
cob_gpm = n_gpm.getInfo() / esperadas_gpm
print(f'Cobertura de imágenes GPM en el periodo: {100 * cob_gpm:.1f}% de lo esperado')
assert cob_gpm >= 0.95, 'GPM no cubre el periodo completo: la comparación no sería justa'

sumas_gpm = [lluvia_gpm_mm(LLUVIA_INI - pd.DateOffset(years=k), LLUVIA_FIN - pd.DateOffset(years=k))[0]
             for k in range(1, ANIOS_CLIMA + 1)]
gpm_normal = ee.ImageCollection.fromImages(sumas_gpm).mean().rename('gpm_normal_mm')

reg_g = gpm_actual.addBands(gpm_normal).reduceRegion(
    reducer=ee.Reducer.mean(), geometry=region, scale=ESCALA_GPM, maxPixels=1e9).getInfo()
GPM_ACTUAL_REGIONAL_MM, GPM_NORMAL_REGIONAL_MM = reg_g['gpm_mm'], reg_g['gpm_normal_mm']
print(f'Zona GPM: actual {GPM_ACTUAL_REGIONAL_MM:.1f} mm | normal {GPM_NORMAL_REGIONAL_MM:.1f} mm')

# GPM tiene píxeles de ~11 km: se toma el píxel del centroide (promediar dentro del predio devuelve vacío)
filas = []
for i in range(0, len(cent), TAMANO_LOTE):
    fc = geemap.geopandas_to_ee(cent.iloc[i:i + TAMANO_LOTE])
    out = gpm_actual.addBands(gpm_normal).sampleRegions(collection=fc, properties=['CODIGO'], scale=ESCALA_GPM)
    filas += [x['properties'] for x in out.getInfo()['features']]
df_gpm_eval = pd.DataFrame(filas)
df_gpm_eval['CODIGO'] = df_gpm_eval['CODIGO'].astype(str).str.strip()
df_gpm_eval = df_gpm_eval.drop_duplicates('CODIGO')
assert len(df_gpm_eval) == len(gdf), 'Faltan predios sin dato GPM'
df_gpm_eval['Amenaza_GPM_pixel'] = (1 - df_gpm_eval['gpm_mm'] / df_gpm_eval['gpm_normal_mm']).clip(0, 1)
df_gpm_eval['Amenaza_GPM_regional'] = (1 - df_gpm_eval['gpm_mm'] / GPM_NORMAL_REGIONAL_MM).clip(0, 1)
df_gpm_eval.to_csv(f'{RUTA_BASE}/repel_amenaza_gpm_evaluacion.csv', index=False)
print(df_gpm_eval[['gpm_mm', 'gpm_normal_mm', 'Amenaza_GPM_pixel', 'Amenaza_GPM_regional']].describe().round(3))

In [ ]:
with open(f'{RUTA_BASE}/parametros_modelo_chirps_{FECHA_FIN}.json', encoding='utf-8') as fh:
    params_modelo = json.load(fh)
LLUVIA_ACTUAL_REGIONAL_MM = params_modelo['lluvia_actual_zona_mm']
LLUVIA_NORMAL_REGIONAL_MM = params_modelo['lluvia_normal_zona_mm']

def clasificar_riesgo(score):
    if pd.isna(score): return 'Sin dato'
    if score > 0.75: return 'Crítico'
    if score > 0.50: return 'Alto'
    if score > 0.25: return 'Moderado'
    return 'Bajo'

comp = (
    gdf[['CODIGO', 'V_Vulnerability', 'precip_mm', 'Amenaza_Norm']]
    .rename(columns={'precip_mm': 'chirps_mm', 'Amenaza_Norm': 'Amenaza_CHIRPS'})
    .merge(df_gpm_eval[['CODIGO', 'gpm_mm', 'Amenaza_GPM_pixel', 'Amenaza_GPM_regional']], on='CODIGO', how='inner')
)
assert len(comp) == len(gdf) == 1848
for src in ['CHIRPS', 'GPM_pixel', 'GPM_regional']:
    comp[f'Riesgo_{src}'] = PESO_AMENAZA * comp[f'Amenaza_{src}'] + (1 - PESO_AMENAZA) * comp['V_Vulnerability']
    comp[f'Nivel_{src}'] = comp[f'Riesgo_{src}'].apply(clasificar_riesgo)

print('Lluvia del periodo (mm): media por centroide de predio')
print(comp[['chirps_mm', 'gpm_mm']].describe().round(1))
print(f'\nZona: CHIRPS actual {LLUVIA_ACTUAL_REGIONAL_MM:.1f} / normal {LLUVIA_NORMAL_REGIONAL_MM:.1f} mm | '
      f'GPM actual {GPM_ACTUAL_REGIONAL_MM:.1f} / normal {GPM_NORMAL_REGIONAL_MM:.1f} mm')

print('\nAmenaza: estadísticos')
est = comp[['Amenaza_GPM_regional', 'Amenaza_GPM_pixel', 'Amenaza_CHIRPS']].agg(['mean', 'std', 'min', 'max', 'nunique']).round(3)
display(est)

print('\nPredios por nivel con cada fuente (V, pesos y umbrales idénticos)')
orden = ['Crítico', 'Alto', 'Moderado', 'Bajo']
display(pd.DataFrame({k: comp[f'Nivel_{k}'].value_counts() for k in ['GPM_regional', 'GPM_pixel', 'CHIRPS']}).reindex(orden).fillna(0).astype(int))

print('\nTransiciones GPM (normal regional, método v2.2) -> CHIRPS')
display(pd.crosstab(comp['Nivel_GPM_regional'], comp['Nivel_CHIRPS'], margins=True))

print('\nCorrelación entre amenazas (Pearson / Spearman):')
print(comp[['Amenaza_GPM_pixel', 'Amenaza_CHIRPS']].corr().round(3).iloc[0, 1], '/',
      round(comp['Amenaza_GPM_pixel'].corr(comp['Amenaza_CHIRPS'], method='spearman'), 3))
print('Correlación de orden del RIESGO (Spearman):',
      round(comp['Riesgo_GPM_regional'].corr(comp['Riesgo_CHIRPS'], method='spearman'), 3))
top = lambda c: set(comp.nlargest(int(len(comp) * 0.1), c)['CODIGO'])
print('Coincidencia en el 10 % más riesgoso:',
      f"{100 * len(top('Riesgo_GPM_regional') & top('Riesgo_CHIRPS')) / int(len(comp) * 0.1):.0f} %")

comp.to_csv(f'{RUTA_BASE}/comparacion_gpm_vs_chirps_{f(FIN)}.csv', index=False)
print('\nLectura obligatoria: la fuente de lluvia cambia sustancialmente la clasificación. '
      'No hay estaciones IDEAM para decidir cuál es más cercana a la realidad. '
      'Los predios que pasan a Alto con CHIRPS no se interpretan como estrés confirmado.')

## Fin del notebook 2

**Límites:** NASA POWER es un punto meteorológico, no una medición por predio; el escenario de yuca usa déficit diario `max(ETc-P,0)` sin almacenamiento de agua en el suelo. Las comparaciones entre satélites no validan por sí solas cuál producto es más exacto.


## 6. Comparación exploratoria de tres cultivos: yuca, maíz y sorgo

**Objetivo:** contrastar exposición climática potencial, **no** predecir rendimientos ni recomendar un cultivo para un predio. Los tres cultivos están documentados en las EVA de Repelón. Se conserva intacto el análisis anterior de yuca.

**Supuestos provisionales para probar el método:** las duraciones y los coeficientes Kc de maíz y sorgo son escenarios de trabajo, **pendientes de cotejar con fuentes agronómicas y variedades locales antes de publicar conclusiones**. La yuca conserva los parámetros ya usados en este notebook (210 días). Se simulan siembras el 1 de abril, 1 de mayo y 1 de junio de 2018–2025.

El indicador diario `max(ETc - P, 0)` no representa el balance hídrico del suelo; ignora almacenamiento, infiltración, riego y escorrentía. La comparación usa el mismo punto NASA POWER para todos los cultivos. Se muestran indicadores por día para evitar comparar solo acumulados de ciclos de distinta duración.

In [ ]:
# COMPARACION DE CULTIVOS — parametros exploratorios editables
# No interpretar como Kc validados para las variedades de Repelon.
import numpy as np
import pandas as pd
from pathlib import Path

# (dias_inicial, dias_desarrollo, dias_media, dias_final), (Kc_ini, Kc_mid, Kc_fin)
CULTIVOS = {
    'Yuca':  {'etapas': (20, 40, 90, 60), 'kc': (0.30, 0.80, 0.30)},
    'Maiz':  {'etapas': (20, 35, 40, 25), 'kc': (0.30, 1.20, 0.60)},
    'Sorgo': {'etapas': (20, 30, 40, 30), 'kc': (0.30, 1.00, 0.55)},
}
MESES_SIEMBRA = (4, 5, 6)

# Validacion basica de la fuente meteorologica y su cobertura.
met = df_et0[['FECHA', 'ET0_mm_dia']].merge(
    df_power_diario[['FECHA', 'PRECTOTCORR']], on='FECHA', validate='one_to_one'
).rename(columns={'PRECTOTCORR': 'P_mm_dia'})
met['FECHA'] = pd.to_datetime(met['FECHA'])
assert met['FECHA'].is_unique and met['ET0_mm_dia'].ge(0).all()
assert met['P_mm_dia'].ge(0).all(), 'Revisar lluvia NASA POWER negativa o faltante'
assert met[['ET0_mm_dia', 'P_mm_dia']].notna().all().all()
met = met.set_index('FECHA').sort_index()

def curva_kc(etapas, coeficientes):
    a, b, c, d = etapas
    ki, km, kf = coeficientes
    return np.r_[np.full(a, ki), np.linspace(ki, km, b+1)[1:],
                 np.full(c, km), np.linspace(km, kf, d+1)[1:]]

filas = []
for cultivo, config in CULTIVOS.items():
    kc = curva_kc(config['etapas'], config['kc'])
    for anio in range(2018, 2026):
        for mes in MESES_SIEMBRA:
            inicio = pd.Timestamp(anio, mes, 1)
            fechas = pd.date_range(inicio, periods=len(kc), freq='D')
            if not fechas.isin(met.index).all():
                raise ValueError(f'Serie incompleta para {cultivo}, {inicio.date()}')
            periodo = met.loc[fechas]
            et0 = periodo['ET0_mm_dia'].to_numpy()
            p = periodo['P_mm_dia'].to_numpy()
            etc = et0 * kc
            deficit = np.maximum(etc-p, 0)
            filas.append({
                'cultivo': cultivo, 'anio': anio, 'siembra': str(inicio.date()),
                'duracion_dias': len(kc), 'lluvia_mm': round(p.sum(), 2),
                'etc_mm': round(etc.sum(), 2),
                'deficit_climatico_mm': round(deficit.sum(), 2),
                'deficit_mm_dia': round(deficit.mean(), 3),
                'fraccion_dias_deficit': round(np.mean(deficit > 0), 4),
                'deficit_20pct_menos_lluvia_mm_dia': round(np.maximum(etc-0.8*p, 0).mean(), 3),
                'deficit_20pct_mas_lluvia_mm_dia': round(np.maximum(etc-1.2*p, 0).mean(), 3),
                'kc_escenario': 'PROVISIONAL - pendiente de validacion documental'
            })

comparacion_cultivos = pd.DataFrame(filas)
assert len(comparacion_cultivos) == 3*8*3
resumen_comparacion = (comparacion_cultivos.groupby(['cultivo', 'siembra'], as_index=False)
    .agg({'deficit_mm_dia':'mean'}))
comparacion_por_mes = (comparacion_cultivos.assign(mes_siembra=lambda d: pd.to_datetime(d.siembra).dt.month)
    .groupby(['cultivo', 'mes_siembra'], as_index=False)
    .agg(deficit_medio_mm_dia=('deficit_mm_dia','mean'),
         fraccion_dias_deficit=('fraccion_dias_deficit','mean'),
         deficit_menos_lluvia=('deficit_20pct_menos_lluvia_mm_dia','mean'),
         deficit_mas_lluvia=('deficit_20pct_mas_lluvia_mm_dia','mean'),
         ciclos=('anio','count')).round(3))
print('Comparacion exploratoria por cultivo y mes de siembra:')
display(comparacion_por_mes)
print('Total escenarios:', len(comparacion_cultivos))

# Solo dos archivos finales de este modulo, en la misma carpeta existente de Drive.
Path(RUTA_BASE).mkdir(parents=True, exist_ok=True)
comparacion_cultivos.to_csv(f'{RUTA_BASE}/comparacion_3_cultivos_escenarios.csv', index=False)
comparacion_por_mes.to_csv(f'{RUTA_BASE}/comparacion_3_cultivos_resumen.csv', index=False)
print('Guardados en Drive: comparacion_3_cultivos_escenarios.csv y comparacion_3_cultivos_resumen.csv')

### Interpretación pendiente

Antes de redactar conclusiones sobre qué cultivo conviene explorar: (1) verificar Kc y duración por cultivo con fuentes técnicas, (2) comprobar consistencia de las comparaciones bajo cambios de fechas y lluvia, (3) no equiparar menor déficit por día con mayor rendimiento o rentabilidad, y (4) conservar el contraste IDEAM como evidencia de incertidumbre meteorológica. La comparación **no se une matemáticamente** al índice territorial ni permite recomendar cultivos a nivel predial.

## 7. Auditoría documental y robustez de los parámetros agronómicos

**Referencias técnicas:** FAO-56 (Allen et al., 1998), tablas 11 y 12, https://www.fao.org/4/x0490e/x0490e00.htm ; FAO *Crop Water Needs*, https://www.fao.org/4/s2022e/s2022e07.htm ; FAO sobre yuca https://www.fao.org/4/x5791e/x5791e0q.htm .

**Hallazgo:** los Kc originales de maíz y sorgo son aproximaciones plausibles pero deben especificar destino de cosecha (grano seco vs fresco); la yuca de 210 días es un ciclo **corto hipotético**, no un ciclo agronómico validado para Repelón. FAO describe yuca de 9–24 meses en una referencia general. No se extrapola la evidencia de FAO a variedades locales sin campo.

**Prueba:** se conservan los 72 escenarios anteriores para trazabilidad. Aquí se comparan escenarios de duración (maíz 120/140/160; sorgo 120/130/140; yuca 210/270/330 días), Kc medio y final (±15%), y lluvia (±20%). La prueba es exploratoria y no representa distribuciones probabilísticas. Los resultados de duración de yuca más largos pueden requerir datos hasta 2026; se omiten ciclos sin datos completos, documentando cuántos se incluyen. El ranking se evalúa **emparejando mismo año, mes y perturbación**, no comparando promedios de distintas condiciones. Menor déficit climático diario no implica mayor productividad ni tolerancia fisiológica.


In [ ]:
# AUDITORIA DE ROBUSTEZ: Kc, duracion y lluvia; no reemplaza validacion de campo.
from itertools import product
import numpy as np
import pandas as pd

# Escenarios tecnicos razonables para prueba, NO ciclos locales confirmados.
DURACIONES = {'Yuca': (210,270,330), 'Maiz': (120,140,160), 'Sorgo': (120,130,140)}
# FAO-56: valores orientativos para maiz y sorgo de grano.
# Yuca: Kc de trabajo a contrastar; su ciclo corto no es una recomendacion.
KC_REFERENCIA = {'Yuca': (0.30,0.80,0.30), 'Maiz': (0.30,1.20,0.35), 'Sorgo': (0.30,1.05,0.45)}
FACTORES_KC = (0.85, 1.00, 1.15)
FACTORES_LLUVIA = (0.8, 1.0, 1.2)

def distribuir_etapas(etapas_base, duracion):
    proporciones = np.asarray(etapas_base, dtype=float)/sum(etapas_base)
    partes = np.floor(proporciones*duracion).astype(int)
    for j in np.argsort(-(proporciones*duracion-partes))[:duracion-sum(partes)]:
        partes[j] += 1
    assert sum(partes)==duracion and min(partes)>0
    return tuple(partes)

robustez_filas = []
for cultivo, base in CULTIVOS.items():
    for duracion, factor_kc, factor_p in product(DURACIONES[cultivo], FACTORES_KC, FACTORES_LLUVIA):
        etapas = distribuir_etapas(base['etapas'], duracion)
        ki, km, kf = KC_REFERENCIA[cultivo]
        kc = curva_kc(etapas, (ki, km*factor_kc, kf*factor_kc))
        for anio in range(2018,2026):
            for mes in MESES_SIEMBRA:
                inicio = pd.Timestamp(anio,mes,1)
                fechas = pd.date_range(inicio,periods=duracion)
                if not fechas.isin(met.index).all():
                    continue
                periodo = met.loc[fechas]
                et0 = periodo['ET0_mm_dia'].to_numpy()
                lluvia = periodo['P_mm_dia'].to_numpy()*factor_p
                deficit = np.maximum(et0*kc-lluvia,0)
                robustez_filas.append(dict(cultivo=cultivo,anio=anio,mes_siembra=mes,
                    duracion_dias=duracion,factor_kc=factor_kc,factor_lluvia=factor_p,
                    deficit_mm_dia=deficit.mean(),fraccion_dias_deficit=(deficit>0).mean()))
robustez = pd.DataFrame(robustez_filas)
assert not robustez.empty
# Comparar cada cultivo en condiciones equivalentes; duracion varia como factor independiente.
medias = (robustez.groupby(['cultivo','anio','mes_siembra','factor_kc','factor_lluvia'],as_index=False)
          .agg(deficit_mm_dia=('deficit_mm_dia','mean'),n_duraciones=('duracion_dias','nunique')))
pareados = medias.pivot(index=['anio','mes_siembra','factor_kc','factor_lluvia'],columns='cultivo',values='deficit_mm_dia').dropna()
pareados['menor_deficit'] = pareados[['Yuca','Maiz','Sorgo']].idxmin(axis=1)
frecuencia = (pareados['menor_deficit'].value_counts().rename_axis('cultivo')
              .reset_index(name='escenarios_ganados'))
frecuencia['porcentaje'] = (100*frecuencia['escenarios_ganados']/len(pareados)).round(1)
print('Escenarios individuales:',len(robustez),'| Comparaciones emparejadas:',len(pareados))
print('Ganador de menor deficit diario bajo sensibilidad:')
display(frecuencia)
print('Rangos por cultivo (todos los supuestos):')
display(robustez.groupby('cultivo')['deficit_mm_dia'].agg(['min','median','max']).round(3))
print('Cobertura de escenarios por cultivo/duracion:')
display(robustez.groupby(['cultivo','duracion_dias'])[['anio','mes_siembra']].size().rename('simulaciones').reset_index())
# Diagnostico critico: resultado yuca vs maiz/sorgo segun condiciones
for otro in ['Maiz','Sorgo']:
    diferencia = pareados['Yuca']-pareados[otro]
    print(f'Yuca - {otro}: mediana {diferencia.median():.3f} mm/dia; fraccion con yuca menor: {(diferencia<0).mean():.1%}')
# No sobrescribir los CSV previos: un archivo nuevo solo para auditoria.
robustez.to_csv(f'{RUTA_BASE}/comparacion_3_cultivos_robustez.csv',index=False)
print('Exportado: comparacion_3_cultivos_robustez.csv')


## 8. Auditoría final: comparabilidad y sensibilidad independiente

Esta sección **no reemplaza** los resultados anteriores: comprueba sus limitaciones.

- Usa únicamente fechas de siembra con cobertura completa para **todas** las duraciones de los tres cultivos.
- Compara combinaciones independientes de duración y Kc para cada cultivo, bajo la **misma lluvia** por fecha de siembra. Esto es una prueba más exigente que variar el Kc de todos simultáneamente.
- Distingue dos estimandos: **déficit medio diario** (normalizado por duración) y **déficit acumulado por ciclo** (no directamente comparable entre duraciones).
- Los escenarios son una prueba de sensibilidad, no una estimación de probabilidades agronómicas; cada combinación recibe el mismo peso arbitrario.
- La precipitación NASA POWER representa condiciones climáticas del punto de referencia, no mediciones de agua disponible en cada parcela.

**Criterio de cierre:** informar cobertura, porcentaje de victorias y excepciones de la yuca; si la clasificación cambia, explicar bajo qué supuestos. No inferir rendimiento, aptitud predial ni recomendaciones validadas en campo.


In [ ]:
# AUDITORÍA FINAL: comparaciones completas y Kc independientes.
# Ejecutar después de las secciones 6 y 7; reutiliza met, curva_kc,
# distribuir_etapas, CULTIVOS, DURACIONES, KC_REFERENCIA y MESES_SIEMBRA.
from itertools import product
import numpy as np
import pandas as pd

# 1. Intersección de fechas con TODOS los ciclos completos, sin truncamiento.
fechas_comunes = []
for anio in range(2018, 2026):
    for mes in MESES_SIEMBRA:
        inicio = pd.Timestamp(anio, mes, 1)
        if all(pd.date_range(inicio, periods=d).isin(met.index).all()
               for ds in DURACIONES.values() for d in ds):
            fechas_comunes.append((anio, mes, inicio))
print('Fechas de siembra originales:', 8 * len(MESES_SIEMBRA))
print('Fechas con cobertura completa para todos los ciclos:', len(fechas_comunes))
assert fechas_comunes, 'No hay fechas comunes; revisar cobertura de NASA POWER.'

# 2. Para cada cultivo, simular 3 duraciones × 3 Kc × 3 lluvias,
#    con las mismas fechas de siembra. No se redondea antes de comparar.
filas_audit = []
for anio, mes, inicio in fechas_comunes:
    for cultivo, base in CULTIVOS.items():
        for duracion, fk, fp in product(DURACIONES[cultivo], FACTORES_KC, FACTORES_LLUVIA):
            etapas = distribuir_etapas(base['etapas'], duracion)
            ki, km, kf = KC_REFERENCIA[cultivo]
            kc = curva_kc(etapas, (ki, km*fk, kf*fk))
            datos = met.loc[pd.date_range(inicio, periods=duracion)]
            et0 = datos['ET0_mm_dia'].to_numpy()
            lluvia = datos['P_mm_dia'].to_numpy() * fp
            deficit = np.maximum(et0*kc - lluvia, 0)
            filas_audit.append({
                'cultivo': cultivo, 'anio': anio, 'mes_siembra': mes,
                'duracion_dias': duracion, 'factor_kc': fk, 'factor_lluvia': fp,
                'deficit_mm_dia': float(deficit.mean()),
                'deficit_total_mm': float(deficit.sum()),
                'fraccion_dias_deficit': float((deficit>0).mean())
            })
audit = pd.DataFrame(filas_audit)
assert audit.groupby(['cultivo','duracion_dias']).size().nunique() == 1
print('Escenarios completos por cultivo:', audit.groupby('cultivo').size().to_dict())

# 3. Comparación exhaustiva: Kc y duración varían INDEPENDIENTEMENTE
#    para cada cultivo; el factor de lluvia es común a los tres.
#    9 alternativas/cultivo -> 9^3=729 combinaciones por fecha y lluvia.
filas_comp = []
for (anio, mes, fp), g in audit.groupby(['anio','mes_siembra','factor_lluvia']):
    a = {c: g[g.cultivo==c].reset_index(drop=True) for c in DURACIONES}
    assert all(len(v)==9 for v in a.values())
    for iy, im, is_ in product(range(9), repeat=3):
        dy = a['Yuca'].iloc[iy]['deficit_mm_dia']
        dm = a['Maiz'].iloc[im]['deficit_mm_dia']
        ds = a['Sorgo'].iloc[is_]['deficit_mm_dia']
        ganador = min([('Yuca',dy),('Maiz',dm),('Sorgo',ds)], key=lambda t:t[1])[0]
        filas_comp.append({
            'anio': anio, 'mes_siembra': mes, 'factor_lluvia': fp,
            'ganador': ganador, 'yuca_menos_maiz': dy-dm,
            'yuca_menos_sorgo': dy-ds,
            'duracion_yuca': int(a['Yuca'].iloc[iy]['duracion_dias']),
            'duracion_maiz': int(a['Maiz'].iloc[im]['duracion_dias']),
            'duracion_sorgo': int(a['Sorgo'].iloc[is_]['duracion_dias']),
            'factor_kc_yuca': float(a['Yuca'].iloc[iy]['factor_kc']),
            'factor_kc_maiz': float(a['Maiz'].iloc[im]['factor_kc']),
            'factor_kc_sorgo': float(a['Sorgo'].iloc[is_]['factor_kc']),
        })
comparaciones_audit = pd.DataFrame(filas_comp)
assert len(comparaciones_audit)==len(fechas_comunes)*3*729

print('\nCOMPARACIÓN EXHAUSTIVA — Kc y duración independientes:')
print('Comparaciones:',len(comparaciones_audit))
print('Ganadores (%):')
display((comparaciones_audit.ganador.value_counts(normalize=True)*100).round(2).rename('porcentaje').to_frame())
print('Yuca menor que maíz:', f"{(comparaciones_audit.yuca_menos_maiz < 0).mean():.1%}")
print('Yuca menor que sorgo:', f"{(comparaciones_audit.yuca_menos_sorgo < 0).mean():.1%}")
print('\nGanadores por mes de siembra (%):')
display((pd.crosstab(comparaciones_audit.mes_siembra, comparaciones_audit.ganador,
                    normalize='index')*100).round(1))
print('\nEjemplos de escenarios donde NO gana la yuca:')
display(comparaciones_audit[comparaciones_audit.ganador!='Yuca'].head(8))
print('\nDéficit diario (mm/día) por cultivo, cobertura completa:')
display(audit.groupby('cultivo')['deficit_mm_dia'].agg(['min','median','max']).round(3))

# 4. Guardar dos archivos NUEVOS, sin modificar los CSV anteriores.
audit.to_csv(f'{RUTA_BASE}/comparacion_3_cultivos_auditoria_cobertura.csv',index=False)
comparaciones_audit.to_csv(f'{RUTA_BASE}/comparacion_3_cultivos_auditoria_pareada.csv',index=False)
print('\nExportados: auditoria_cobertura.csv y auditoria_pareada.csv')



# 9. Six crops and three sequential rotations (24 months)

**New experimental module, not field validation.** Uses the `met` daily NASA POWER table and the existing `curva_kc` function above. Do not run this cell until all earlier notebook cells have executed. This module does not overwrite previous CSV files.

Six crops: cassava, grain maize, sorghum, dry beans (generic, not confirmed cowpea), tomato, and flooded rice. Parameters are illustrative references, not cultivar-specific Repelón prescriptions. The flooded rice case includes FAO illustrative saturation, water-layer and percolation requirements, which must not be interpreted as measured district irrigation needs.

Three two-year sequences are *proposed*, not observed farm rotations. No soil fertility, pest control, yields, economics, or irrigation availability are predicted.


In [ ]:
# SIX-CROP PARAMETERS — FAO reference values and explicitly provisional stage durations.
# FAO irrigation manual: https://www.fao.org/4/s2022e/s2022e07.htm
# FAO rice-specific irrigation: https://www.fao.org/4/s2022e/s2022e08.htm
# Original three crops preserve their prior baseline assumptions; this module
# is a NEW scenario family and does not silently replace earlier comparisons.
from pathlib import Path
import json
import numpy as np
import pandas as pd

CULTIVOS_6 = {
 'Cassava': {'family':'Euphorbiaceae','stages':(20,40,90,60),'kc':(.30,.80,.30),'status':'provisional existing model'},
 'Maize':   {'family':'Poaceae','stages':(20,35,40,25),'kc':(.30,1.20,.60),'status':'provisional existing model'},
 'Sorghum': {'family':'Poaceae','stages':(20,30,40,30),'kc':(.30,1.00,.55),'status':'provisional existing model'},
 'Dry bean':{'family':'Fabaceae','stages':(20,25,35,20),'kc':(.35,1.10,.30),'status':'FAO indicative; generic dry bean, not local cowpea'},
 'Tomato':  {'family':'Solanaceae','stages':(30,35,40,25),'kc':(.45,1.15,.80),'status':'FAO indicative; local variety unverified'},
 'Rice':    {'family':'Poaceae','stages':(25,30,40,25),'kc':(1.05,1.20,.90),'status':'FLOODED rice illustrative; Kc and water regime unverified'},
}
assert set(CULTIVOS_6)=={'Cassava','Maize','Sorghum','Dry bean','Tomato','Rice'}
assert met.index.is_unique and met.index.is_monotonic_increasing
assert met[['ET0_mm_dia','P_mm_dia']].notna().all().all()
print(pd.DataFrame([{'crop':k,'family':v['family'],'days':sum(v['stages']), 'Kc':v['kc'],'status':v['status']} for k,v in CULTIVOS_6.items()]).to_string(index=False))


In [ ]:
# CROP-SPECIFIC DAILY CLIMATIC SHORTFALL; rice additionally reports hypothetical irrigation.
# Note: max(ETc - rainfall, 0) per day does NOT carry soil-water storage forward.
# For flooded rice, SAT=200 mm and WL=100 mm once per cycle, PERC=6 mm/day.
# FAO illustrative quantities: not measured at Repelón; sensitivity 4–8 mm/day.

def evaluar_ciclo(crop, start, rainfall_factor=1.0, rice_perc=6.0):
    cfg=CULTIVOS_6[crop]
    days=sum(cfg['stages'])
    dates=pd.date_range(pd.Timestamp(start),periods=days,freq='D')
    if not dates.isin(met.index).all():return None
    x=met.loc[dates]
    kc=curva_kc(cfg['stages'],cfg['kc'])
    assert len(kc)==days
    rain=x.P_mm_dia.to_numpy(dtype=float)*rainfall_factor
    etc=x.ET0_mm_dia.to_numpy(dtype=float)*kc
    deficit=np.maximum(etc-rain,0)
    result={'crop':crop,'start':str(dates[0].date()),'end':str(dates[-1].date()),
            'days':days,'rain_mm':float(rain.sum()),'etc_mm':float(etc.sum()),
            'shortfall_mm':float(deficit.sum()),'shortfall_mm_day':float(deficit.mean()),
            'days_shortfall_pct':float(100*(deficit>0).mean()),
            'family':cfg['family'],'rice_extra_water_mm':None,'rice_irrigation_proxy_mm':None}
    if crop=='Rice':
        # FAO scheme illustrative; uses gross rainfall as an optimistic proxy for effective rainfall.
        # Distinct from the daily climate shortfall; do NOT rank directly against other crops.
        extra=200+100+rice_perc*days
        result['rice_extra_water_mm']=float(extra)
        result['rice_irrigation_proxy_mm']=float(max(0,etc.sum()+extra-rain.sum()))
    return result

rows=[]
for year in range(2018,2026):
 for month in range(1,13):
  for crop in CULTIVOS_6:
   for factor in (.8,1.,1.2):
    r=evaluar_ciclo(crop,f'{year}-{month:02d}-01',factor)
    if r:rows.append({**r,'rain_factor':factor,'year':year,'month':month})
crop6=pd.DataFrame(rows)
assert not crop6.empty and not crop6.duplicated(['crop','start','rain_factor']).any()
print('Complete six-crop cycles:',len(crop6))
display(crop6.groupby('crop').agg(cycles=('start','count'),median_shortfall=('shortfall_mm_day','median')).round(2))


In [ ]:
# 24-MONTH ROTATIONS: successive crops on the same hypothetical field.
# The 14-day gap is an explicit management assumption, NOT a modeled cover crop.
# Dates automatically follow the preceding harvest; no overlaps allowed.
ROTATION_TEMPLATES={
 'A': ['Maize','Dry bean','Cassava'],
 'B': ['Sorghum','Tomato','Rice'],
 'C': ['Rice','Dry bean','Maize','Cassava'],
}
GAP_DAYS=14

def evaluar_rotacion(name, first_date, factor=1.0, rice_perc=6.0):
    begin=pd.Timestamp(first_date);end_limit=begin+pd.DateOffset(months=24)
    cursor=begin;events=[]
    for crop in ROTATION_TEMPLATES[name]:
        r=evaluar_ciclo(crop,cursor,factor,rice_perc)
        if r is None:return None
        if pd.Timestamp(r['end'])>=end_limit:return None
        events.append(r)
        cursor=pd.Timestamp(r['end'])+pd.Timedelta(days=1+GAP_DAYS)
    # Count the last 14-day gap as unused land, not a crop, and no gap after final harvest.
    occupied=sum(r['days'] for r in events)
    horizon=(end_limit-begin).days
    assert occupied<=horizon
    for i in range(1,len(events)):
        assert pd.Timestamp(events[i]['start'])>pd.Timestamp(events[i-1]['end'])
    # Mean deficit per CROPPED day; NOT mean over 24 months.
    return {'rotation':name,'start':str(begin.date()),'end_horizon':str((end_limit-pd.Timedelta(days=1)).date()),
            'rain_factor':factor,'crops':' → '.join(ROTATION_TEMPLATES[name]),
            'families':len({e['family'] for e in events}),
            'cropped_days':occupied,'unmodeled_days':horizon-occupied,
            'climate_shortfall_total_mm':sum(e['shortfall_mm'] for e in events),
            'climate_shortfall_per_cropped_day':sum(e['shortfall_mm'] for e in events)/occupied,
            'rice_irrigation_proxy_mm':sum(e['rice_irrigation_proxy_mm'] or 0 for e in events),
            'rice_extra_water_mm':sum(e['rice_extra_water_mm'] or 0 for e in events),
            'segments':events}

rotation_rows=[]
for year in range(2018,2024):
 for month in (4,5,6):
  for factor in (.8,1.,1.2):
   for name in ROTATION_TEMPLATES:
    r=evaluar_rotacion(name,f'{year}-{month:02d}-01',factor)
    if r:rotation_rows.append(r)
rotations24=pd.DataFrame(rotation_rows)
assert not rotations24.empty
print('Complete rotation scenarios:',len(rotations24))
display(rotations24.groupby('rotation').agg(n=('start','count'),median_deficit=('climate_shortfall_per_cropped_day','median'),median_cropped_days=('cropped_days','median')).round(2))
print('IMPORTANT: rice irrigation proxy is separate and cannot be compared as equivalent to climatic shortfall.')


In [ ]:
# EXPORT: never overwrite previous notebook outputs.
# All results are generated locally/Drive on execution, not precomputed in this notebook.
from pathlib import Path
out=Path(RUTA_BASE)/'rotaciones_6_cultivos'
out.mkdir(parents=True,exist_ok=True)
crop6.to_csv(out/'six_crop_climate_scenarios.csv',index=False)
rotations24.drop(columns='segments').to_csv(out/'three_rotations_24_months.csv',index=False)
segments=[]
for r in rotation_rows:
 for i,e in enumerate(r['segments'],1):
  segments.append({'rotation':r['rotation'],'start_horizon':r['start'],'rain_factor':r['rain_factor'],
                   'sequence':i,**e})
pd.DataFrame(segments).to_csv(out/'rotation_segments.csv',index=False)
metadata={'sources':{'Kc':'https://www.fao.org/4/s2022e/s2022e07.htm',
                      'rice':'https://www.fao.org/4/s2022e/s2022e08.htm'},
          'limitations':['No measured crop yields or field rotation validation',
             'Generic crop stages and Kc are not locally calibrated',
             'NASA POWER is regional point data, not parcel irrigation',
             'Daily rainfall deficit ignores soil-water carryover',
             'Flooded rice requires additional water; rice proxy is separate',
             'Gaps are unmodeled, not quantified cover crop benefits']}
(out/'assumptions.json').write_text(json.dumps(metadata,indent=2),encoding='utf-8')
# For offline HTML dashboard: a JS file with compact baseline summaries and rotation details.
base_crop=crop6[crop6.rain_factor==1].groupby(['crop','month'],as_index=False).agg(
 mean_deficit=('shortfall_mm_day','mean'),n=('shortfall_mm_day','size'))
base_rot=rotations24[rotations24.rain_factor==1].copy()
summary=base_rot.groupby('rotation',as_index=False).agg(
 deficit=('climate_shortfall_per_cropped_day','mean'),
 n=('climate_shortfall_per_cropped_day','size'),
 families=('families','first'),cropped_days=('cropped_days','first'),
 rice_extra=('rice_extra_water_mm','first'))
package={'crops':base_crop.round(3).to_dict(orient='records'),
         'rotations':summary.round(3).to_dict(orient='records'),
         'templates':ROTATION_TEMPLATES,'limitations':metadata['limitations']}
(out/'rotation_results.js').write_text('window.ROTATION_RESULTS='+json.dumps(package,ensure_ascii=False)+';',encoding='utf-8')
print('EXPORTED:',out)
print('Copy rotation_results.js beside index.html, data.js, geo.js and crop_results.js after execution.')



### Interpretation / publication gate

- The previous three-crop sensitivity result (66.01% cassava wins) **does not extend to six crops or rotations**. Recompute any ranking from the exported files.
- For rotation A/B/C, compare climatic shortfall per cropped day **alongside** cropped days, family diversity, rice extra water and the unmodeled time. A single score is not justified.
- Generic dry bean is **not proof** of suitability for local cowpea; rice assumes flooded management and local irrigation access must be checked.
- Crop rotations are hypothetical two-year sequences, not observed local practices or verified recommendations.
